# Week 11 — Deep Learning Foundations & ANN
### Building and Training an Artificial Neural Network in PyTorch
**Next Gen Software Hub Pvt. Ltd. — AI & ML Internship**


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} | PyTorch version: {torch.__version__}")


## 1. Data Ingestion & Tensor Conversion


In [ ]:
df = pd.read_csv("datasets/sensor_fault_detection.csv")
X = df.drop(columns=["Fault_Detected"]).values
y = df["Fault_Detected"].values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

train_dataset = TensorDataset(torch.FloatTensor(X_train_scaled), torch.FloatTensor(y_train).unsqueeze(1))
val_dataset = TensorDataset(torch.FloatTensor(X_val_scaled), torch.FloatTensor(y_val).unsqueeze(1))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


## 2. Define the Neural Network Architecture (MLP)


In [ ]:
class SensorFaultANN(nn.Module):
    def __init__(self, input_dim):
        super(SensorFaultANN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

model = SensorFaultANN(input_dim=X_train.shape[1]).to(device)
print(model)


## 3. Training Loop with Loss Tracking


In [ ]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

epochs = 40
train_losses = []
val_losses = []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_x.size(0)
    
    epoch_train_loss = running_loss / len(train_dataset)
    train_losses.append(epoch_train_loss)
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(device), by.to(device)
            preds = model(bx)
            l = criterion(preds, by)
            val_loss += l.item() * bx.size(0)
            
    epoch_val_loss = val_loss / len(val_dataset)
    val_losses.append(epoch_val_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:2d}/{epochs} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")


## 4. Plot Loss Curves


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(train_losses, label="Training Loss", color="royalblue")
plt.plot(val_losses, label="Validation Loss", color="coral", linestyle="--")
plt.title("ANN Training vs Validation Loss Curve", fontsize=14, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.legend()
plt.show()
